# 00_02 Notebook Basics

## Goal
Learn how Databricks notebooks work: cells, magic commands, widgets, display, and running one notebook from another.

## Why it matters
Real pipelines are built from many notebooks. Knowing magic commands and `%run` lets you mix SQL and Python and reuse code instead of copying it.

## Concept
A notebook is a list of cells. Each cell runs in the notebook's default language (Python here), unless its first line is a **magic command**:

| Magic | Purpose |
|---|---|
| `%python` | Run Python |
| `%sql` | Run SQL |
| `%md` | Markdown text |
| `%run` | Run another notebook in this notebook's context |

The magic command must be the **first line** of the cell.

## Real-world example
An ETL notebook cleans data in Python, then an analyst runs a `%sql` cell on the same data to check results. A shared `config` notebook is pulled in with `%run`.

# 2. Notebook basics

## 2.1 print vs show vs display

**What it does:** creates a small DataFrame and shows it three ways.
**Why it matters:** `print` shows the object description, `show()` prints a text table, `display()` gives an interactive table with sorting and charts.
**Expected output:** a schema-like description, a text table, then an interactive table.

In [0]:
data = [(1, "Amal", 21000), (2, "Vanathaiyan", 15000)]
df = spark.createDataFrame(data, ["id", "name", "salary"])

print(df)       # prints the object description, not the data
df.show()       # plain text table
display(df)     # interactive Databricks table (sort, chart, download)

## 2.2 Temp view: bridge between Python and SQL

**What it does:** registers the DataFrame as a temporary view named `employees`.
**Why it matters:** once a view exists, you can query the same data with SQL, so you can mix Python and SQL in one notebook.
**Expected output:** no output. The view now exists for this session.

In [0]:
df.createOrReplaceTempView("employees")

## 2.3 SQL magic command

**What it does:** runs a SQL query on the `employees` view using `%sql`.
**Why it matters:** `%sql` must be the first line of the cell. It lets analysts use SQL on data prepared in Python.
**Expected output:** one row, Amal with 21000.

In [0]:
%sql
SELECT name,salary
FROM employees
WHERE salary > 16000

## 2.4 Widgets (parameters)

**What it does:** creates a text box at the top of the notebook and filters the DataFrame by its value.
**Why it matters:** widgets turn a notebook into a parameterized one. Jobs use the same mechanism to pass values like dates or environment names.
**Expected output:** a widget appears at the top. Change it to `Raj`, re-run, and the result changes.

In [0]:
dbutils.widgets.text("employee_name","Amal","Employee Name")
name = dbutils.widgets.get("employee_name")
display(df.filter(df.name == name))

## 2.5 Remove widgets

**What it does:** deletes all widgets from the notebook.
**Why it matters:** keeps the notebook clean and avoids old widget values affecting later runs.

In [0]:
dbutils.widgets.removeAll()

## 2.6 Reusing code with %run

**What it does:** loads the notebook `00_helper` into this notebook, so its functions become available here.
**Why it matters:** `%run` is how you share helper code and config across notebooks without copying it. It must be alone in its cell.
**Expected output:** no output from the `%run` cell, then `Hello, Databricks!` from the next cell.

In [0]:
%run ./00_helper

In [0]:
print(greet("Databricks"))

## Troubleshooting log

| Problem | Fix |
|---|---|
| `%sql` cell runs as Python and errors | The magic command must be on the first line of the cell |
| `%run` fails or says the notebook is not found | Check the relative path, and keep `%run` alone in its cell |
| | |

## Interview Q&A

1. **What is the difference between `show()` and `display()`?**
   `show()` is a Spark method that prints a text table. `display()` is Databricks-specific and gives an interactive table with sorting and charts.

2. **How do you share data between Python and SQL cells?**
   Register a temp view with `createOrReplaceTempView` in Python, then query it in a `%sql` cell.

3. **What is the difference between `%run` and `dbutils.notebook.run`?**
   `%run` includes the other notebook's code and variables in the current notebook. `dbutils.notebook.run` starts it as a separate run with its own context and returns a value.

4. **What are widgets used for?**
   Parameterizing notebooks, for example passing a date or environment name in from a job.

## Practice

1. Create a DataFrame of 5 products with `id`, `name` and `price`. Register a temp view and query it with `%sql` for products above a given price.
2. Add a widget `min_price` and use it to filter the DataFrame in Python.
3. Move your DataFrame-creating code into a helper notebook and load it with `%run`.

## Summary

- A magic command must be the first line of a cell
- `display()` is interactive, `show()` is plain text
- Temp views bridge Python and SQL
- Widgets parameterize notebooks
- `%run` reuses code from another notebook

